# Notebook 13  
## Protein ESM-2 Embedding Extraction (HiPerGator)

This notebook extracts pretrained transformer embeddings for protein sequences
using Meta AI’s ESM-2 model.

Pipeline:

Protein sequence  
→ ESM-2 tokenizer  
→ Transformer encoder  
→ Mean pooling over tokens  
→ Fixed-length embedding vector  

These embeddings will later be used for classical ML baselines
(LogReg, SVM, XGBoost) and hybrid models.

This notebook only performs embedding extraction and saving.

In [1]:
import torch
import esm
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [2]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [5]:
# Paths
DATA_PATH = Path("../data/processed/protein_uniprot_pfam_top10_per400.csv")
SAVE_DIR = Path("../data/processed")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
df = pd.read_csv(DATA_PATH)
df = df[["accession", "sequence", "family"]]

df.head()

,accession,sequence,family
0,Q96RD1,MRNHTEITEFILLGLTDDPNFQVVIFVFLLITYMLSITGNLTLITI...,PF13853
1,Q9H210,MRQINQTQVTEFLLLGLSDGPHTEQLLFIVLLGVYLVTVLGNLLLI...,PF13853
2,Q8NGZ3,MNHSVVTEFIILGLTKKPELQGIIFLFFLIVYLVAFLGNMLIIIAK...,PF13853
3,O60412,MERGNQTEVGNFLLLGFAEDSDMQLLLHGLFLSMYLVTIIGNLLII...,PF13853
4,Q8NGS0,MENQSSISEFFLRGISAPPEQQQSLFGIFLCMYLVTLTGNLLIILA...,PF13853


## Load ESM-2 Model

We use esm2_t12_35M_UR50D as a strong pretrained baseline.

In [7]:
model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
model = model.to(device)
model.eval()

batch_converter = alphabet.get_batch_converter()

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t12_35M_UR50D.pt" to /home/dpratapa/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t12_35M_UR50D-contact-regression.pt" to /home/dpratapa/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D-contact-regression.pt


## Embedding Extraction Function

In [8]:
def extract_embeddings(sequences, batch_size=16):
    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), batch_size)):
            batch_seqs = sequences[i:i+batch_size]
            batch_data = [(str(idx), seq) for idx, seq in enumerate(batch_seqs)]
            _, _, batch_tokens = batch_converter(batch_data)

            batch_tokens = batch_tokens.to(device)

            outputs = model(batch_tokens, repr_layers=[12], return_contacts=False)
            token_representations = outputs["representations"][12]

            # Mean pooling (exclude padding + special tokens)
            for j, seq in enumerate(batch_seqs):
                seq_len = len(seq)
                emb = token_representations[j, 1:seq_len+1].mean(0)
                embeddings.append(emb.cpu().numpy())

    return np.array(embeddings)

In [9]:
sequences = df["sequence"].tolist()

embeddings = extract_embeddings(sequences, batch_size=8)

embeddings.shape

100%|██████████| 287/287 [00:55<00:00,  5.19it/s]


(2293, 480)

In [10]:
# Encode labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
labels = le.fit_transform(df["family"])

labels.shape

(2293,)

In [11]:
np.save(SAVE_DIR / "protein_esm2_embeddings_top10_per400.npy", embeddings)
np.save(SAVE_DIR / "protein_esm2_labels_top10_per400.npy", labels)

print("Saved embeddings and labels.")

Saved embeddings and labels.


In [16]:
import numpy as np

emb_path = "/home/dpratapa/Capstone/data/processed/protein_esm2_embeddings_top10_per400.npy"

X_emb = np.load(emb_path)

print("Shape:", X_emb.shape)
print("Dtype:", X_emb.dtype)

Shape: (2293, 480)
Dtype: float32


In [17]:
print("Mean:", np.mean(X_emb))
print("Std:", np.std(X_emb))
print("Min:", np.min(X_emb))
print("Max:", np.max(X_emb))
print("Any NaNs:", np.isnan(X_emb).any())
print("Any Infs:", np.isinf(X_emb).any())

Mean: 0.0050942255
Std: 0.2220985
Min: -2.435789
Max: 6.147586
Any NaNs: False
Any Infs: False


In [18]:
import numpy as np

labels = np.load("/home/dpratapa/Capstone/data/processed/protein_esm2_labels_top10_per400.npy")

unique, counts = np.unique(labels, return_counts=True)

print("Counts per class:", counts)
print("All equal to 400?", np.all(counts == 400))

Counts per class: [287 187 203 125 121 194 350 309 117 400]
All equal to 400? False


In [19]:
import pandas as pd
from collections import Counter

df = pd.read_csv("/home/dpratapa/Capstone/data/processed/protein_uniprot_pfam_top10_per400.csv")

print(Counter(df["family"]))

Counter({'PF13853': 400, 'PF01352': 350, 'PF07686': 309, 'PF00001': 287, 'PF00069': 203, 'PF00096': 194, 'PF00046': 187, 'PF00071': 125, 'PF00076': 121, 'PF12796': 117})


In [21]:
import pandas as pd

df3 = pd.read_csv("/home/dpratapa/Capstone/data/processed/protein_uniprot_pfam_top10_per400.csv")

print(df3.columns)
print(df3.head())

Index(['accession', 'sequence', 'length', 'family'], dtype='str')
  accession                                           sequence  length  \
0    Q96RD1  MRNHTEITEFILLGLTDDPNFQVVIFVFLLITYMLSITGNLTLITI...     312   
1    Q9H210  MRQINQTQVTEFLLLGLSDGPHTEQLLFIVLLGVYLVTVLGNLLLI...     308   
2    Q8NGZ3  MNHSVVTEFIILGLTKKPELQGIIFLFFLIVYLVAFLGNMLIIIAK...     307   
3    O60412  MERGNQTEVGNFLLLGFAEDSDMQLLLHGLFLSMYLVTIIGNLLII...     319   
4    Q8NGS0  MENQSSISEFFLRGISAPPEQQQSLFGIFLCMYLVTLTGNLLIILA...     311   

    family  
0  PF13853  
1  PF13853  
2  PF13853  
3  PF13853  
4  PF13853  
